# 01: Advanced Data Cleaning, Outlier Remediation & Imputation

**Track 02: Data Analytics, EDA & High-Performance Dataframes** | *Tensorbox AI/ML Production Curriculum*

---
### Overview & Objectives
Production data quality engineering: Missing value strategies (KNNImputer, IterativeImputer), IQR and Z-Score outlier clipping, type casting, and schema validation.


## 1. Ingesting Real-World Data via Tensorbox Data Loader
Let's load the Titanic and Housing datasets with automatic local caching and Kaggle sync.

In [ ]:
import os
import sys
from pathlib import Path

for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (p / "utils").exists():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        break

from utils.data_loader import load_dataset

df_raw = load_dataset("titanic")
print(f"Loaded Titanic Dataset: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
print(df_raw.head())

## 2. Missing Value Profiling & Multi-Strategy Imputation
Comparing Simple Median Imputation against KNN and Iterative Multivariable Imputation (MICE).

In [ ]:
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

print("Missing Values Per Column:")
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0])

# Impute numerical features using KNN
num_cols = ["Age", "Fare", "SibSp", "Parch"]
knn_imputer = KNNImputer(n_neighbors=5)
df_imputed = df_raw.copy()
df_imputed[num_cols] = knn_imputer.fit_transform(df_raw[num_cols])

# Impute categorical 'Embarked' with Mode
df_imputed["Embarked"] = df_imputed["Embarked"].fillna(df_imputed["Embarked"].mode()[0])

print("\nAfter Imputation Missing Values Check:")
print(df_imputed[num_cols + ['Embarked']].isnull().sum())

## 3. Outlier Detection: IQR vs Z-Score Filtering
Detect and remediate extreme anomalies in financial and price data.

In [ ]:
def iqr_outlier_clipping(series, factor=1.5):
    q25 = series.quantile(0.25)
    q75 = series.quantile(0.75)
    iqr = q75 - q25
    lower_bound = q25 - (factor * iqr)
    upper_bound = q75 + (factor * iqr)
    clipped = series.clip(lower=max(0, lower_bound), upper=upper_bound)
    return clipped, lower_bound, upper_bound

fare_clean, low, high = iqr_outlier_clipping(df_imputed["Fare"])
print(f"Fare Outlier Thresholds: Lower=${low:.2f}, Upper=${high:.2f}")
print(f"Original Fare Max: ${df_imputed['Fare'].max():.2f} -> Clipped Fare Max: ${fare_clean.max():.2f}")
df_imputed["Fare_Clean"] = fare_clean